# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and perform exploratory data analysis (EDA) with the FAIR² clinical oncology dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is structured and described by a [Croissant schema](https://mlcommons.org/croissant/) available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running on a fresh environment)
!pip install mlcroissant --quiet

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview

Let's examine which record sets are present in the dataset, and see available fields (columns) in each.

- **Record sets** typically correspond to tables or primary entities in the dataset.
- **Fields** are the expected data attributes/columns, identified by their unique `@id`s.

We’ll reference entities using their Croissant `@id` values for clarity and reproducibility.

In [ ]:
# List all record sets with their @id and name
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset metadata.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"  - {rs['@id']}")
        print(f"    Name: {rs.get('name', 'N/A')}")
        if 'field' in rs:
            print("    Fields:")
            for fld in rs['field']:
                if isinstance(fld, dict):
                    print(f"      - {fld.get('@id', str(fld))}")
                else:
                    print(f"      - {str(fld)}")
        elif 'fields' in rs:  # fallback, rarely used
            print("    Fields:")
            for fld in rs['fields']:
                if isinstance(fld, dict):
                    print(f"      - {fld.get('@id', str(fld))}")
                else:
                    print(f"      - {str(fld)}")
        else:
            print("    No fields listed.")
        print()

## 3. Data Extraction

Load records from a specific record set into a pandas DataFrame. For this dataset, we'll attempt to find the main record set and load all records from it. We reference by `@id` as shown above.

**Note:** If the dataset is structured with only one main record set (a single table), we use that one. Otherwise, we demonstrate on the first available.

In [ ]:
# For demonstration, load all available record sets
dataframes = {}
if not record_sets:
    print("No tabular record sets available.")
else:
    print("Loading data from the available record sets...")
    for rs in record_sets:
        rs_id = rs['@id']
        try:
            records = list(dataset.records(record_set=rs_id))
            if records:
                dataframes[rs_id] = pd.DataFrame(records)
                print(f"Record set '{rs_id}' loaded: {len(records)} records. Columns:")
                print(dataframes[rs_id].columns.tolist())
            else:
                print(f"Record set '{rs_id}': No rows found.")
        except Exception as e:
            print(f"Error loading records for record set '{rs_id}': {e}")

# If a main record set is found, select it for further exploration
if dataframes:
    main_record_set_id = list(dataframes.keys())[0] # Choose the first loaded
    print(f"\nContinuing analysis using record set: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes to analyze.")

## 4. Exploratory Data Analysis (EDA)

Let's apply some typical EDA steps:
- Filter records by a numeric field (e.g., age, interval, or other count fields)
- Normalize that numeric field
- Optionally, group by a categorical/grouping variable (e.g., sex, cancer type)

We'll use the field `@id` when referencing columns.

In [ ]:
# Find a numeric field in the dataframe automatically if possible
import numpy as np

df = dataframes[main_record_set_id]

# Display some info to help find a numeric column
print("Available columns:")
for c in df.columns:
    print(c)

# Try to find a numeric column
numeric_field = None
for col in df.columns:
    # Try to infer if column is numeric
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    # Try a heuristic: columns which look like 'age', 'interval', 'count',...'years'
    for col in df.columns:
        if any(s in col.lower() for s in ['age', 'interval', 'count', 'years', 'duration', 'value']):
            # Try converting to numeric
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break

if numeric_field is None:
    print("No obvious numeric field found to demonstrate EDA.")
else:
    print(f"Using numeric field (@id): {numeric_field}")
    threshold = np.nanpercentile(df[numeric_field], 75)  # e.g., upper quartile as example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df[[col for col in filtered_df.columns if col == numeric_field or pd.api.types.is_numeric_dtype(filtered_df[col])]].head())
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
    print(f"Normalized field '{numeric_field}':")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find a group field (e.g., 'sex', 'gender', 'site', 'type')
    group_field = None
    for s in ['sex', 'gender', 'location', 'site', 'type', 'status', 'msi']:
        for col in df.columns:
            if s in col.lower():
                group_field = col
                break
        if group_field:
            break
    # Group by that field
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No obvious group field for aggregation found.")

## 5. Visualization

Let's visualize the distribution of our chosen numeric variable, and if available, its breakdown by a group category (e.g., by MSI status or anatomical site).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is None:
    print("No numeric field found, skipping visualization.")
else:
    plt.figure(figsize=(7, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
    
    # If group_field exists, boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We loaded the FAIR² clinical oncology dataset using its Croissant schema with `mlcroissant`.
- We identified available record sets and fields using their `@id`s, and loaded the main data into a DataFrame.
- We performed basic EDA: filtering by a numeric field, normalization, and grouping by a categorical field (when available).
- Finally, visualizations provided a quick insight into the distribution and potential differences by category.

For more advanced analysis, see [mlcroissant documentation](https://mlcommons.github.io/croissant/python/index.html) and consult the dataset's Croissant schema for additional metadata and relationships.

**Remember:** All field and entity references above rely on their Croissant `@id` for transparency and reproducibility.